# 12 - Red profunda multimodal con fusión por compuertas (gated fusion)

Modelo no clásico: ramas por modalidad (texto/video/sensorial) + compuertas aprendidas + HPO de arquitectura y optimización.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)


In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for c in candidates:
        if (c / 'materials' / 'dataset_task3_exist2026').exists() and (c / 'notebooks_shiyi').exists():
            return c / 'notebooks_shiyi'
    if (cwd / 'artifacts').exists() and (cwd / 'entregables').exists():
        return cwd
    raise FileNotFoundError('No se pudo resolver notebooks_shiyi.')

PROJECT_ROOT = resolve_project_root()
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
PRED_DIR = PROJECT_ROOT / 'predicciones_finales'
WEIGHTS_DIR = PROJECT_ROOT / 'weights_hpo'
REPORTS_DIR = PROJECT_ROOT / 'reports_hpo'
for d in [PRED_DIR, WEIGHTS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FUSION_TRAIN_PATH = ARTIFACTS_DIR / 'fusion_features_train.npz'
FUSION_TEST_PATH = ARTIFACTS_DIR / 'fusion_features_test.npz'

train_bundle = np.load(FUSION_TRAIN_PATH, allow_pickle=True)
test_bundle = np.load(FUSION_TEST_PATH, allow_pickle=True)

X_train_all = train_bundle['X_fusion'].astype(np.float32)
y_train_all = train_bundle['y'].astype(np.int64)
langs_train_all = train_bundle['langs'].astype(str)
sample_ids_train = train_bundle['sample_ids'].astype(str)

X_test_all = test_bundle['X_fusion'].astype(np.float32)
y_test_all = test_bundle['y'].astype(np.int64)
langs_test_all = test_bundle['langs'].astype(str)
sample_ids_test = test_bundle['sample_ids'].astype(str)

TEST_JSON_PATH = PROJECT_ROOT.parent / 'materials' / 'dataset_task3_exist2026' / 'test.json'
if not TEST_JSON_PATH.exists():
    raise FileNotFoundError(f'test.json not found: {TEST_JSON_PATH}')
with open(TEST_JSON_PATH, 'r', encoding='utf-8') as f:
    test_raw = json.load(f)
TEST_IDS = {str(k) for k in test_raw.keys()}
test_id_mask_test = np.array([sid in TEST_IDS for sid in sample_ids_test], dtype=bool)

text_dim = int(train_bundle['text_dim'][0])
video_dim = int(train_bundle['video_dim'][0])
sensor_dim = int(train_bundle['sensor_dim'][0])

X_text_train = X_train_all[:, :text_dim]
X_video_train = X_train_all[:, text_dim:text_dim + video_dim]
X_sensor_train = X_train_all[:, text_dim + video_dim:text_dim + video_dim + sensor_dim]

X_text_test = X_test_all[:, :text_dim]
X_video_test = X_test_all[:, text_dim:text_dim + video_dim]
X_sensor_test = X_test_all[:, text_dim + video_dim:text_dim + video_dim + sensor_dim]

print('dims => text:', text_dim, 'video:', video_dim, 'sensor:', sensor_dim)
print('train rows:', X_train_all.shape[0], '| test rows:', X_test_all.shape[0], '| test ids matched:', int(test_id_mask_test.sum()))


In [ ]:
class GatedMultimodalNet(nn.Module):
    def __init__(self, text_dim, video_dim, sensor_dim, hidden=256, dropout=0.3):
        super().__init__()
        self.text_branch = nn.Sequential(
            nn.Linear(text_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU()
        )
        self.video_branch = nn.Sequential(
            nn.Linear(video_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU()
        )
        self.sensor_branch = nn.Sequential(
            nn.Linear(sensor_dim, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, hidden // 2), nn.ReLU()
        )

        fuse_dim = (hidden // 2) * 3
        self.gate = nn.Sequential(
            nn.Linear(fuse_dim, fuse_dim),
            nn.Sigmoid()
        )
        self.classifier = nn.Sequential(
            nn.Linear(fuse_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )

    def forward(self, x_text, x_video, x_sensor):
        zt = self.text_branch(x_text)
        zv = self.video_branch(x_video)
        zs = self.sensor_branch(x_sensor)
        z = torch.cat([zt, zv, zs], dim=1)
        g = self.gate(z)
        z = z * g
        logits = self.classifier(z).squeeze(1)
        return logits


In [ ]:
def train_one(model, train_loader, val_loader, epochs=30, lr=1e-3, weight_decay=1e-4, pos_weight=1.0):
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best = {'f1': -1.0, 'state': None, 'thr': 0.5}

    for _ in range(epochs):
        model.train()
        for xb_t, xb_v, xb_s, yb in train_loader:
            xb_t = xb_t.to(DEVICE)
            xb_v = xb_v.to(DEVICE)
            xb_s = xb_s.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb_t, xb_v, xb_s)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        all_probs, all_true = [], []
        with torch.no_grad():
            for xb_t, xb_v, xb_s, yb in val_loader:
                logits = model(xb_t.to(DEVICE), xb_v.to(DEVICE), xb_s.to(DEVICE))
                probs = torch.sigmoid(logits).cpu().numpy()
                all_probs.append(probs)
                all_true.append(yb.numpy())

        probs = np.concatenate(all_probs)
        y_true = np.concatenate(all_true).astype(int)

        best_t, best_f1 = 0.5, -1.0
        for t in np.linspace(0.2, 0.8, 121):
            y_pred = (probs >= t).astype(int)
            f1 = f1_score(y_true, y_pred, average='macro')
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)

        if best_f1 > best['f1']:
            best['f1'] = float(best_f1)
            best['thr'] = float(best_t)
            best['state'] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best['state'])
    return best['f1'], best['thr']

def make_loader(Xt, Xv, Xs, y, batch_size=64, shuffle=True):
    ds = TensorDataset(
        torch.tensor(Xt, dtype=torch.float32),
        torch.tensor(Xv, dtype=torch.float32),
        torch.tensor(Xs, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


In [ ]:
def export_json(sample_ids, pred_binary, output_path):
    rows = [
        {'test_case': 'EXIST2026', 'id': str(sid), 'value': 'YES' if int(v) == 1 else 'NO'}
        for sid, v in zip(sample_ids, pred_binary)
    ]
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False)

def run_config(tag, train_langs):
    mask = np.isin(langs_train_all, train_langs) & (y_train_all >= 0)
    Xt = X_text_train[mask]
    Xv = X_video_train[mask]
    Xs = X_sensor_train[mask]
    y = y_train_all[mask]

    if len(np.unique(y)) < 2:
        raise RuntimeError(f'{tag}: subset sin dos clases.')

    Xtr_t, Xva_t, Xtr_v, Xva_v, Xtr_s, Xva_s, ytr, yva = train_test_split(
        Xt, Xv, Xs, y, test_size=0.2, random_state=SEED, stratify=y
    )

    space = [
        {'hidden': 256, 'dropout': 0.25, 'lr': 1e-3, 'wd': 1e-4, 'bs': 64, 'epochs': 35},
        {'hidden': 384, 'dropout': 0.30, 'lr': 7e-4, 'wd': 1e-4, 'bs': 64, 'epochs': 40},
        {'hidden': 512, 'dropout': 0.35, 'lr': 5e-4, 'wd': 5e-5, 'bs': 96, 'epochs': 45},
        {'hidden': 384, 'dropout': 0.20, 'lr': 1.5e-3, 'wd': 1e-5, 'bs': 128, 'epochs': 35}
    ]

    pos = float((ytr == 0).sum() / max((ytr == 1).sum(), 1))
    trials = []
    best = {'f1': -1.0}

    for cfg in space:
        model = GatedMultimodalNet(text_dim, video_dim, sensor_dim, hidden=cfg['hidden'], dropout=cfg['dropout']).to(DEVICE)
        tr_loader = make_loader(Xtr_t, Xtr_v, Xtr_s, ytr, batch_size=cfg['bs'], shuffle=True)
        va_loader = make_loader(Xva_t, Xva_v, Xva_s, yva, batch_size=cfg['bs'], shuffle=False)
        f1_val, thr = train_one(model, tr_loader, va_loader, epochs=cfg['epochs'], lr=cfg['lr'], weight_decay=cfg['wd'], pos_weight=pos)

        row = dict(cfg)
        row.update({'config': tag, 'val_f1_macro': float(f1_val), 'best_thr': float(thr)})
        trials.append(row)

        if f1_val > best['f1']:
            best = {
                'f1': float(f1_val),
                'thr': float(thr),
                'cfg': cfg,
                'state': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            }

    rep = pd.DataFrame(trials).sort_values('val_f1_macro', ascending=False)
    rep_path = REPORTS_DIR / f'deep_gated_hpo_{tag}.csv'
    rep.to_csv(rep_path, index=False)

    final_model = GatedMultimodalNet(text_dim, video_dim, sensor_dim, hidden=best['cfg']['hidden'], dropout=best['cfg']['dropout']).to(DEVICE)
    final_model.load_state_dict(best['state'])
    final_model.eval()

    infer_mask = test_id_mask_test & np.isin(langs_test_all, train_langs)
    if not infer_mask.any():
        raise RuntimeError(f'{tag}: no test rows for selected langs')

    with torch.no_grad():
        logits_test = final_model(
            torch.tensor(X_text_test[infer_mask], dtype=torch.float32, device=DEVICE),
            torch.tensor(X_video_test[infer_mask], dtype=torch.float32, device=DEVICE),
            torch.tensor(X_sensor_test[infer_mask], dtype=torch.float32, device=DEVICE)
        )
        probs_test = torch.sigmoid(logits_test).cpu().numpy()
        pred_test = (probs_test >= best['thr']).astype(int)

    model_path = WEIGHTS_DIR / f'best_deep_gated_{tag}.pt'
    torch.save({
        'state_dict': best['state'],
        'cfg': best['cfg'],
        'thr': best['thr'],
        'dims': {'text': text_dim, 'video': video_dim, 'sensor': sensor_dim}
    }, model_path)

    print(f"{tag} => val_f1={best['f1']:.4f}, thr={best['thr']:.3f}, cfg={best['cfg']}")
    print(classification_report(yva, (torch.sigmoid(final_model(torch.tensor(Xva_t, dtype=torch.float32, device=DEVICE), torch.tensor(Xva_v, dtype=torch.float32, device=DEVICE), torch.tensor(Xva_s, dtype=torch.float32, device=DEVICE))).cpu().numpy() >= best['thr']).astype(int), digits=4))

    out_path = PRED_DIR / f'BeingChillingWeWillWin_DEEPGATED_{tag}.json'
    export_json(sample_ids_test[infer_mask], pred_test, out_path)

run_config('EN', ['en'])
